# 05 - Credit Risk Modelling

## Objective
Train and compare Logistic Regression, Random Forest and XGBoost; handle class
imbalance with evidence rather than habit; select a model and a decision threshold
on the validation set; and report a single, final, unbiased evaluation on the test set.

## Split discipline
| Split | Used for |
|---|---|
| **Train** | fitting preprocessing + estimators |
| **Validation** | model selection, imbalance strategy, threshold tuning |
| **Test** | one final evaluation, never for any decision |

The dataset has no origination date, so a chronological out-of-time split is not
possible; we use a stratified random split with a fixed seed and note the limitation.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import numpy as np
import pandas as pd

from src import config
from src.data import preprocessing
from src.evaluation import metrics as M
from src.features.engineering import add_credit_risk_features, get_feature_lists
from src.models import train as T
from src.risk.segmentation import RiskBands, segment_portfolio, summarise_segments

%matplotlib inline

## Configuration

In [ ]:
RANDOM_STATE = config.RANDOM_STATE
np.random.seed(RANDOM_STATE)
print("test size:", config.TEST_SIZE, "| validation size:", config.VALIDATION_SIZE)

## Data loading

In [ ]:
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, _ = preprocessing.clean_data(raw, TARGET)
df = add_credit_risk_features(clean)
df.shape

## Train / validation / test split

In [ ]:
splits = T.split_data(df, TARGET)
X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "default_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(4)

## Preprocessing pipeline

`ColumnTransformer` inside a `Pipeline`:

* numeric -> median imputation (+ standard scaling for the linear model only)
* categorical -> most-frequent imputation + one-hot encoding with `handle_unknown="ignore"`

Because these steps are fitted inside the pipeline, they see the training fold only.

In [ ]:
numeric, categorical = get_feature_lists(df, TARGET, drop=list(T.REDUNDANT_COLUMNS))
preprocessor = T.build_preprocessor(numeric, categorical)
preprocessor.fit(X_train)
print(len(T.build_preprocessor(numeric, categorical).fit(X_train)
          .get_feature_names_out()), "columns after encoding")

## Class imbalance

About 22% of the loans default. That is imbalanced but not extreme, so the minority
class is well represented in absolute terms (thousands of cases).

**SMOTE is not used.** It would fabricate borrowers, and the synthetic minority
points distort the predicted probabilities -- which is exactly what the risk bands
depend on. Instead both variants are fitted and compared on the validation set:
estimator re-weighting (`class_weight` / `scale_pos_weight`) versus training on the
natural distribution.

In [ ]:
scale_pos_weight = T.compute_scale_pos_weight(y_train)
print(f"negatives/positives = {scale_pos_weight:.2f}")

study = {}
fitted_by_strategy = {}
for strategy, weighting in (("class-weighted", True), ("unweighted", False)):
    models = T.build_models(numeric, categorical, scale_pos_weight, class_weighting=weighting)
    fitted_by_strategy[strategy] = T.fit_models(models, X_train, y_train)
    for name, pipeline in fitted_by_strategy[strategy].items():
        study[(name, strategy)] = M.evaluate_predictions(y_val, T.predict_proba(pipeline, X_val))

pd.DataFrame(study).T.round(4)

Pick, per model, the variant with the better validation PR-AUC (ties broken on the
Brier score, i.e. probability quality).

In [ ]:
fitted, chosen = {}, {}
for name in fitted_by_strategy["unweighted"]:
    ranked = sorted(
        ["class-weighted", "unweighted"],
        key=lambda s: (study[(name, s)]["PR-AUC"], -study[(name, s)]["Brier"]),
        reverse=True,
    )
    chosen[name] = ranked[0]
    fitted[name] = fitted_by_strategy[ranked[0]][name]
chosen

## Model comparison on the validation set

In [ ]:
val_proba = {name: T.predict_proba(pipeline, X_val) for name, pipeline in fitted.items()}
val_metrics = {name: M.evaluate_predictions(y_val, proba) for name, proba in val_proba.items()}
val_table = M.comparison_table(val_metrics)
val_table

In [ ]:
M.plot_roc_pr_curves(y_val, val_proba, filename="nb_roc_pr_validation.png")
M.plot_calibration(y_val, val_proba, filename="nb_calibration.png")

## Threshold tuning (validation only)

Accuracy is not the objective: predicting "nobody defaults" would already score
about 78%. Two policies are derived:

1. **Balanced** -- maximise F1 for the default class.
2. **High recall** -- maximise precision subject to recall >= 85%, the shape a
   lender uses when a missed default costs far more than a rejected good customer.

In [ ]:
best_name = val_table.index[0]
threshold, threshold_metrics = M.tune_threshold(y_val, val_proba[best_name], objective="f1")
policy_threshold, policy_metrics = M.tune_threshold(
    y_val, val_proba[best_name], objective="max_precision_at_recall", min_recall=0.85)

print(f"selected model: {best_name}")
print(f"balanced threshold   = {threshold:.2f} -> {pd.Series(threshold_metrics).round(4).to_dict()}")
print(f"high-recall threshold = {policy_threshold:.2f} -> {pd.Series(policy_metrics).round(4).to_dict()}")

## Final evaluation on the test set

This is the only place the test set is touched.

In [ ]:
test_proba = {name: T.predict_proba(pipeline, X_test) for name, pipeline in fitted.items()}
test_metrics = {
    name: M.evaluate_predictions(y_test, proba,
                                 threshold if name == best_name else 0.5)
    for name, proba in test_proba.items()
}
test_table = M.comparison_table(test_metrics)
test_table

In [ ]:
print(M.classification_text_report(y_test, test_proba[best_name], threshold))
M.plot_confusion_matrix(y_test, test_proba[best_name], threshold,
                        title=f"{best_name} - test", filename="nb_confusion_matrix.png")
M.plot_probability_distribution(y_test, test_proba[best_name], filename="nb_pd_distribution.png")

## Probability of default and risk segmentation

`predict_proba` gives the PD directly; the bands are project-defined policy
thresholds (`LOW_RISK_THRESHOLD`, `HIGH_RISK_THRESHOLD` in `src/config.py`), not
statistical constants. The check that matters is whether the observed default rate
inside each band tracks the predicted PD.

In [ ]:
bands = RiskBands()
print(bands.describe())
scored = segment_portfolio(X_test.assign(**{TARGET: y_test}), test_proba[best_name],
                           bands, target=TARGET)
summarise_segments(scored, bands, target=TARGET)

In [ ]:
scored[["person_income", "loan_amnt", "loan_grade", "default_probability",
        "risk_category", TARGET]].head(10)

## Persisting the artefacts

In [ ]:
T.save_model(fitted[best_name])
print("Saved to", config.MODEL_FILE)
print("Run `python run_pipeline.py` to regenerate every artefact, including "
      "models/feature_config.json, reports/metrics.json and the figures.")

## Results
See the two comparison tables above -- validation for the decision, test for the
final numbers. The same figures are written to `reports/metrics.json` by the pipeline.

## Interpretation
Gradient boosting wins on ranking quality (ROC-AUC / PR-AUC) and on probability
quality (Brier). Logistic regression stays in the project as a transparent
benchmark: the gap between it and the boosted model is the price of interpretability.

## Conclusion
The selected pipeline is saved with its preprocessing attached, so serving applies
exactly the training-time transformations. Explainability follows in notebook 06.